# 🔬 report6 — 검증 리포트: **우리가 무엇을 틀렸는지 알아낸 기록**

> 이 노트북의 값어치는 *"우리가 옳았다"* 가 아니라 *"무엇이 틀렸는지 알아냈다"* 에 있습니다.
> 다른 리포트가 **결과**를 말한다면, 이 리포트는 **그 결과를 믿어도 되는 이유와, 믿으면 안 되는 부분**을 말합니다.

### 이번에 새로 밝혀진 것 (§2 · §6 · §7)

1. **§2 — "광선을 더 쏘면 되지 않나?"** 4억 발을 쐈습니다. 안 됩니다. 확산 에코의 코히어런트 합은 광선 4배마다 **+8.7 dB** 씩 계속 커지고(수렴하지 않고), 비코히어런트 합은 정답선에서 **-12.7 dB** 떨어진 채 표류합니다. 그리고 값 자체가 **우리가 고르는 노브 S** 의 함수입니다.
   결정적으로, ITU `metal` 은 **S=0** 이라 드론 σ 의 **84%** 를 만드는 모터·배터리·PCB·카메라가 확산 채널에 **기여 0** 입니다. **σ 는 적분에서 나옵니다. GPU 로 해결되지 않습니다.**

2. **§6 — SBR: PO 를 광선추적 *안으로* 넣었습니다.** 상용 EM 솔버(FEKO/CST/HFSS SBR+)가 하는 바로 그것입니다. 광선이 조명면을 찾고(가림 포함), 그 위에서 PO 표면적분을 합니다. **가림이 공짜로 해결됩니다.**
   대가는 큽니다 — 기존 PO 는 RCS 를 **1.5~7.1 dB**, 마이크로도플러 |DC|/std(AC) 를 **17.4 dB** 과대평가하고 있었습니다.
   그리고 오래된 caveat 하나가 **숫자가 됐습니다**: "PO 는 오목부 다중반사를 못 한다" → 재추적해 보니 **-0.33 ~ +0.35 dB**.

3. **§7 — 메쉬·재질 버그 3건.** 기체 외형이 **-47~-2%** 낮았고(σ -0.2~+7.6 dB), 짐벌 카메라를 두 엔진이 **10.9 dB** 다르게 보고 있었으며, 프로펠러 캡 법선이 뒤집혀 있었습니다. 이제 **trimesh 게이트**가 빌드에서 이걸 막습니다.

### 그대로 살아 있는 것 (§2~§5, §8)
RT 에는 표적 σ 가 없다(평판 52 dB 실험) · 디스코볼 · 평판은 PO 커널을 시험하지 못한다 · 널 깊이는 인용 불가 · 챔버는 semi-anechoic · 정적 클러터는 죽은 파라미터 · 표적 경유 바닥 유령.

> ⚠️ **이 리포트에서 절대 하지 않는 주장**: *"레이트레이싱은 RCS 를 못 낸다"* — **거짓**입니다. §6 의 SBR 이 바로 레이트레이싱이고, σ 를 계산합니다.
> 참인 명제는 훨씬 좁습니다: **"산란적분 단계가 없는 전파용 path solver(Sionna RT 기본)에서는 σ 가 창발하지 않는다."**

## 1. 실험 설계 — 무엇을 어떻게 비교했나

**비교량: 직접파 대비 표적에코의 진폭비.** 절대 보정(안테나 이득·송신전력)과 무관해서, **두 엔진을 공정하게 맞댈 수 있는 유일한 양**입니다.

| | 계산 방법 |
|---|---|
| **Sionna RT** | 광선추적 → 표적을 맞고 오는 경로들의 복소이득 합 ÷ 직접파(LOS) 이득 |
| **PO / SBR + 링크버짓** | 바이스태틱 레이더 방정식: ratio = L·√(σ/4π) / (R₁·R₂) |

**기하** (30×20×11 m 챔버, `src/bistatic_scene.py`): L = 15.07 m · R₁ = 18.75 m · R₂ = 18.61 m · β = 47.6° · f = 3.5 GHz
→ 표적에코가 있어야 할 지연 **τ = (R₁+R₂)/c = 124.6 ns**, 직접파 **50.3 ns**.

교정 표적으로 **금속구(r=0.3 m)** 를 골랐습니다 — σ = πr² 로 정답이 알려져 있으니까요.

> ⚠️ **여기서 첫 번째 실수를 했습니다.** *"구는 어떤 각도에서도 정반사점이 있으니 RT 에 가장 유리하다"* 고 생각했는데, **정반사점 탐색(image method) 기준으로는 정확히 반대**였습니다. 구는 RT 에 **가장 불리한** 표적입니다(§4).

## 2. 🆕 "광선을 더 쏘면 되지 않나?" — **4억 발**을 쏴 봤습니다

![ray budget](outputs/figures/report6_ray_budget.png)

가장 자주 나오는 반론이고, **정당한 반론**입니다: *"GPU 가 놀고 있잖아. 광선을 늘리면 σ 가 수렴하지 않겠나?"*
그래서 실제로 늘렸습니다 — `samples_per_src` 를 **25M → 400M (16배)**. 확산(diffuse)을 켜고, 시드 5개로 몬테카를로 잡음까지 분리했습니다.

### (a) 값은 수렴하지 않습니다

| 광선 | 표적경로 수 | 비코히어런트 합 | 코히어런트 합 |
|---|---|---|---|
| 25M | 4 | -75.7 ± 4.2 dB | -72.0 dB |
| 50M | 9 | -74.6 ± 3.5 dB | -66.3 dB |
| 100M | 18 | -73.8 ± 1.4 dB | -62.7 dB |
| 200M | 47 | -72.8 ± 1.2 dB | -57.5 dB |
| 400M | 93 | -72.8 ± 0.6 dB | -54.5 dB |

**정답선은 -60.1 dB 입니다** — 같은 메쉬·같은 재질을 SBR(§6)로 재서 σ = -21.8 dBsm 을 얻고, 그걸 바이스태틱 레이더 방정식에 넣은 값입니다.

- **코히어런트 합**은 광선 4배마다 **+8.7 dB** 씩 커지고 **멈출 기미가 없습니다** (경로 수가 광선 수를 따라 늘고, 그 경로들이 더해지니까요).
- **비코히어런트 합**은 정답선에서 **-12.7 dB** 떨어진 채, 16배를 늘려도 여전히 표류합니다(-75.7 → -72.8 dB, 시드 산포 ±2.2 dB).

> **어느 합산 규약을 쓰든 수렴하지 않습니다.** 수렴할 **목표가 없기 때문**입니다 — 기본 path solver 에는 **표면적분 단계 자체가 없습니다.**

### (b) 게다가 그 값은 **우리가 돌리는 노브**입니다

| 산란계수 S | 0.1 | 0.2 | 0.4 | 0.8 |
|---|---|---|---|---|
| RT 진폭비 | -74.3 dB | -67.1 dB | -60.8 dB | -55.0 dB |

S 를 0.1 → 0.8 로 돌리면 에코가 **+19.3 dB** 움직입니다. 그리고 정답선을 재현하는 값은 **S ≈ 0.44** 입니다.

> **즉 "RT 로 σ 를 맞췄다"는 건 S 를 피팅했다는 뜻입니다.** 예측이 아니라 사후 맞춤입니다.

### (c) 그리고 그 노브는 **정작 중요한 산란체에 닿지도 못합니다**

ITU `metal` 재질은 정의상 **scattering_coefficient = 0** 입니다. 그런데 SBR 로 부품별 기여를 나눠 보면 —

| 부품 | σ 기여 |
|---|---|
| 모터 · 배터리 · PCB · 카메라 하우징 (**S = 0**) | **84%** |
| 셸 · 프로펠러 · 암 (S > 0) | 16% |

**드론 RCS 의 84% 를 만드는 부품들이 RT 의 확산 채널에는 기여가 정확히 0 입니다.** 확산을 아무리 키워도, RT 가 보는 표적과 물리가 보는 표적은 **다른 표적**입니다.

> ## ⇒ σ 는 **표면적분에서 나옵니다. GPU 로 해결되지 않습니다.**
> 그래서 우리가 한 일은 광선을 늘리는 게 아니라, **적분을 광선추적 안에 넣는 것**이었습니다 → §6.

> 재현: `python benchmark/verify_rt_rays.py` (GPU 자동선택)

## 3. RT 의 표적 에코에는 σ 가 들어 있지 않다 (기존 실험, 유지)

![no sigma](outputs/figures/report6_rt_no_sigma.png)

### (a) 정반사 = 무한거울
표적 자리에 **금속 평판**을 놓고 이등분선에 법선을 맞춰(정반사 조건 충족) 변 길이를 **0.2 → 4 m** 로 키웠습니다. 이론 σ 는 4πA²/λ² 이므로 **52 dB 증가**합니다. 그런데 RT 의 진폭비는:

| 평판 변 | 0.2 m | 0.5 m | 1 m | 2 m | 4 m |
|---|---|---|---|---|---|
| 이론 σ | +4.4 | +20.3 | +32.3 | +44.4 | +56.4 |
| **RT 진폭비** | **-7.91** | **-7.91** | **-7.91** | **-7.91** | **-7.91** |

**산포 0.00 dB.** 그리고 그 값은 정확히 image-source(무한거울) 예측 20·log₁₀(L/(R₁+R₂)) = **-7.88 dB** 입니다.
→ Sionna 의 정반사 필드는 **무한평면 거울장**입니다(곡률·유한개구 보정 없음). **정반사 경로가 잡혀도 RCS 정보는 0입니다.**

### (b) 확산 = 물리가 아니라 노브
같은 금속구에서 **산란계수 S 만** 바꾸면 진폭비가 **S² 법칙**으로 이동합니다. 이론값(πr²)에 맞추려면 S ≈ 0.85 가 필요합니다 — §2 의 드론 실험(S ≈ 0.44)과 **같은 결론**입니다: 순환논법입니다.

## 4. 물리적으로 옳은 금속구에서는 경로가 아예 없다 (기존, 유지)

![missing](outputs/figures/report6_rt_missing.png)

ITU `metal` 은 **S = 0** (= PEC, 물리적으로 옳습니다). 그러면 확산 채널이 비어 있고 **정반사만** 가능한데 — **크기를 10배 키우고 표본을 16배 늘려도 표적 경로 0개**입니다. **그런데 같은 자리의 평판은 표본 100만으로도 즉시 잡힙니다** → 메쉬·재질·솔버는 **정상**입니다.

### 원인: 디스코볼 문제
곡면을 삼각형으로 쪼개면 **어떤 패싯도 자기 평면의 정반사점을 자기 삼각형 안에 품지 못합니다.** 세분해도 소용없습니다 — 면 법선의 각도 간격과 면 크기가 **함께** 줄어들어 비율이 그대로거든요. 판별 변수는 **크기가 아니라 곡률**입니다.

> ⚠️ *"곡면에서 정반사는 원리적으로 불가능"* 이라고까지 말하면 안 됩니다. 자세를 무작위로 돌리면 **수 % 확률**로 유효 패싯이 나타나고, 그때는 Sionna 도 경로를 만듭니다. 다만 **잡혀도 진폭은 틀립니다**(단일 패싯이 '평판'으로 반사 → 구 이론 대비 +15 dB). 정확한 표현은 **"저확률 + 잡혀도 진폭 무의미"** 입니다.

## 5. RT 가 **맞히는** 것과, 우리가 **틀렸던** 것

![delay](outputs/figures/report6_rt_delay.png)

처음엔 *"챔버에서 RT 가 PO 대비 −5 dB 로 일치한다"* 고 보고했는데 — **허수였습니다.** 우리 분류 규약(`표적 2 m 이내를 지나면 표적에코`)이 받아들인 경로 **9개** 중 **진짜 1-bounce 표적에코는 1개**뿐이고, 나머지는 **2·3-bounce 벽 반사**였습니다. **좋아 보이는 숫자가 사실은 오분류의 산물**이었습니다.

> **그런데 이 그림은 동시에 RT 의 강점을 증명합니다** — 진짜 에코가 **정확히 (R₁+R₂)/c 자리에** 있습니다.
> **RT 의 기하(지연·도플러)는 정확합니다. 못 주는 건 진폭(σ)뿐입니다.**

## 6. 🆕 **SBR — PO 를 광선추적 *안으로* 넣다**

![sbr](outputs/figures/report6_sbr.png)

§2 가 준 답은 "광선을 늘려라" 가 아니라 **"적분을 넣어라"** 였습니다. 그게 상용 EM 솔버(FEKO / CST / HFSS **SBR+**)가 고주파 RCS 를 내는 표준 방법입니다:

1. **광선(GO)** 으로 *어느 면이 실제로 조명되는가* 를 찾고 — **가림·다중반사가 여기서 해결됩니다**,
2. 그 면들 위에서 **PO 표면적분**을 해서 σ 를 냅니다.

핵심은 변수변환입니다. 모노스태틱 PO 의 비스듬함 계수가 투영면적으로 **상쇄**되므로

> E(û) ∝ ∬_조명면 (n̂·û)·e^{j2k r·û} dS = ∬ e^{j2k r·û} **dA_투영**

즉 **û 방향에서 균일 격자로 광선을 쏘고, 맞은 지점마다 위상을 더하면 그게 PO 적분입니다** (광선 1발 = 투영면적 d²). 구현: `src/rcs_sbr.py` (Mitsuba/OptiX).

### (a) 커널 검증 — 해석해 대조

| 격자 | λ/4 | λ/6 | λ/8 | λ/10 | λ/12 | λ/16 | λ/20 | λ/24 |
|---|---|---|---|---|---|---|---|---|
| 금속구 (πr²) | +3.79 | +1.75 | +0.44 | +0.39 | +1.45 | -0.58 | -0.28 | +0.21 |
| 금속평판 (4πA²/λ²) | +0.29 | -0.01 | -0.17 | -0.26 | -0.01 | -0.17 | +0.11 | +0.14 |

평판은 **어느 격자에서도 정확**하고(|오차| ≤ 0.29 dB), 구는 λ/16 부터 **|오차| ≤ 0.58 dB** 로 수렴합니다.

> ⚠️ **정직하게**: 기본 격자 λ/12 에서 구는 **+1.45 dB** 로 읽힙니다. 이건 계통 편향이 아니라 **곡면 위 광선격자의 이산화 잡음**입니다(격자를 옮기면 부호가 바뀝니다). report3 의 마이크로도플러가 λ/32 격자로 간 것도 같은 이유입니다. **작업 격자에서 커널의 잡음 바닥은 ~1 dB 로 보는 것이 옳습니다.**

### (b) 가림의 대가 — 기존 PO 는 σ 를 과대평가하고 있었다

| 드론 | Mini 5 Pro | Mavic 4 Pro | Matrice 4E | S1000+ | Phantom 4 |
|---|---|---|---|---|---|
| PO (el 15°) | -20.1 | -17.2 | -16.9 | -6.8 | -18.6 |
| **SBR** (el 15°) | **-26.0** | **-21.8** | **-19.6** | **-10.5** | **-23.5** |
| 가림의 효과 | -5.9 dB | -4.6 dB | -2.7 dB | -3.7 dB | -4.9 dB |
| (el 0° 에서) | -7.1 dB | -1.5 dB | -2.4 dB | -3.2 dB | -6.6 dB |

**모든 드론에서 σ 가 내려갑니다 — 1.5 ~ 7.1 dB.** 옛 PO 는 기체에 **가려진 면들까지 더하고 있었습니다.**

### (c) 마이크로도플러는 더 크게 움직입니다

|DC|/std(AC) (Mavic 4 Pro, 3.5 GHz, 5500 rpm): **PO +43.8 dB → SBR +26.3 dB = 17.4 dB**

분해하면 |DC| **-6.0 dB** (가려진 배터리·PCB·블레이드 뒷면이 빠짐) + std(AC) **+11.4 dB** (블레이드가 동체를 가렸다 열며 변조가 **깊어짐**).
→ **마이크로도플러 검출은 옛 PO 추정보다 17.4 dB 쉽습니다.**

### (d) 오래된 caveat 이 숫자가 됐습니다

"PO 는 오목부(로터 아래·짐벌 그늘) 다중반사를 원리적으로 못 다룬다" — 맞습니다. 그래서 **SBR 로 재추적**했습니다:

| 드론 | Mini 5 Pro | Mavic 4 Pro | Matrice 4E | S1000+ | Phantom 4 |
|---|---|---|---|---|---|
| σ(3-bounce) − σ(1-bounce) | +0.28 dB | +0.20 dB | +0.26 dB | -0.33 dB | +0.35 dB |

**-0.33 ~ +0.35 dB.** 이 드론들의 오목부는 λ 스케일에서 얕아서, 다중반사는 **무시할 만합니다.** 리스크 표의 그 줄은 이제 *'모름'* 이 아니라 **'정량화됨'** 입니다.

### (e) 그러나 SBR 도 못 하는 것 — **셸 안쪽**

광선은 **첫 표면에서 멈춥니다.** 그래서 플라스틱 셸 안의 **배터리·PCB 에는 광선이 한 발도 닿지 않습니다**(report2 측정: 적중 0발). 옛 PO 는 그것들을 **가림 없이 전부** 계상했습니다.

> **참값은 SBR(하한)과 PO(상한) 사이에 있습니다.** 어느 쪽이 얼마나 맞는지는 **실측 앵커링 전엔 모릅니다** — §8 리스크 표의 그 줄이 아직 열려 있는 이유입니다.

## 7. 🆕 메쉬·재질 버그 3건 — 그리고 **회귀방지 게이트**

![mesh bugs](outputs/figures/report6_mesh_bugs.png)

엔진(PO→SBR)만 고쳤다면 절반입니다. **입력(메쉬·재질)이 틀려 있었습니다.**

### (1) 드론이 너무 납작했다

파라메트릭 실루엣 규칙(`body_z = 0.35·body_h`)으로 기체를 만들었더니 **공식 제원과 어긋났습니다**:

| | Mini 5 Pro | Mavic 4 Pro | Matrice 4E | S1000+ | Phantom 4 |
|---|---|---|---|---|---|
| 옛 메쉬 높이 | 48.1 mm | 75.8 mm | 93.7 mm | 454.9 mm | 148.2 mm |
| 공식(=수정 후) | 91.0 mm | 135.2 mm | 149.5 mm | 462.0 mm | 198.0 mm |
| 오차 | **-47%** | **-44%** | **-37%** | **-2%** | **-25%** |
| σ 변화 (el 15°) | +4.7 dB | +3.0 dB | +6.2 dB | +0.1 dB | +0.4 dB |
| σ 변화 (el 0°) | +0.9 dB | +4.7 dB | +7.6 dB | -0.2 dB | +3.2 dB |

챔버의 낮은 앙각(el ≈ 15°)에서는 **높이가 측면 투영면적을 지배**합니다. 그래서 이 버그는 RCS 를 **-0.2 ~ +7.6 dB** 움직였습니다.
(위 표의 '옛 메쉬' 는 기억이 아니라 **수정 전 코드 경로를 되살려 다시 만든 메쉬**입니다 — `viz_verify_sbr.build_drone_prefix`.)

### (2) 한 부품, 두 엔진, 두 재질

짐벌 카메라를 **Sionna 는 plastic(|Γ| = 0.244)** 으로, **PO 는 금속 하우징(|Γ| = 0.85)** 으로 보고 있었습니다 — 같은 부품을 **10.9 dB** 다르게 본 겁니다. (드론 σ 에 미치는 영향: **+1.22 dB**)

원인은 재질 체계가 **두 개**였다는 것입니다 — Sionna 용 `RadioMaterial` 과 PO 용 **손으로 적은 |Γ| 표**. 이제 **`materials.py` 하나만** 두고, PO 의 |Γ| 도 거기서 유도합니다. **두 엔진이 조용히 어긋날 수 없습니다.**

### (3) 프로펠러 캡 법선이 뒤집혀 있었다

블레이드 양 끝 캡 2장의 감기가 반대여서 **법선이 안쪽**을 향했습니다(주석엔 "outward" 라고 적혀 있었습니다). PO 는 조명면을 `n̂·û > 0` 로 판정하므로 **프로펠러 672면 중 평균 16면을 잘못 분류**했습니다.

**값을 정직하게 매기면 이 버그는 쌌습니다**: 프로펠러 자체 σ **+0.04 dB**, 드론 전체 **-0.01 dB** (캡은 블레이드의 양 끝면이라 면적이 작습니다). SBR 은 **광선이 맞은 면의 법선을 광선 쪽으로 정렬**하므로 아예 영향이 없습니다(-0.00 dB).

> **그런데도 이 버그가 여기 있는 이유**: **아무도 볼 수 없었기 때문**입니다. 주석은 거짓말을 하고 있었고, 렌더는 멀쩡해 보였고, RCS 숫자는 0.01 dB 만 틀렸습니다. **프로펠러는 마이크로도플러의 신호 그 자체**라, 다음번엔 이렇게 싸지 않았을 겁니다.

### (4) 그래서 게이트를 세웠습니다 — `src/mesh_check.py`

`trimesh` 로 **부품(연결요소) 단위** 검사: watertight · winding 일관성 · **법선 방향(부호있는 부피)** · 퇴화면. `build_all` 이 `assert_ok()` 를 부르므로 **나쁜 메쉬는 빌드를 실패시킵니다.**

| 드론 | Mini 5 Pro | Mavic 4 Pro | Matrice 4E | S1000+ | Phantom 4 |
|---|---|---|---|---|---|
| 부품 / 면 | 25 / 1,572 | 25 / 1,632 | 30 / 1,968 | 49 / 2,860 | 27 / 1,524 |
| 판정 | ✅ PASS | ✅ PASS | ✅ PASS | ✅ PASS | ✅ PASS |

위 (3) 의 버그 메쉬를 넣으면 게이트가 **prop 그룹에서 안쪽법선 부품 4개**를 즉시 잡아냅니다.

## 8. PO 는 믿을 수 있나 — 자기 검증 (기존, 유지)

RT 를 비판했으니 PO 도 같은 잣대로 시험해야 공정합니다. 그런데 여기서 **우리 검증 근거 하나가 무너졌습니다.**

### (a) 평판 검증은 진단력이 0이다
![diagnostic](outputs/figures/report6_po_diagnostic.png)

report2 는 *"금속 평판에서 PO = 4πA²/λ² 이론과 0.00 dB 일치"* 를 검증 근거로 써 왔습니다. 그런데 **수직입사에서는 위상항이 사라집니다(P·û ≡ 0).** 그래서 PO 커널에 **일부러 버그를 심어도** 평판은 그대로 통과합니다 (모든 변종에서 |Δσ| ≈ 1e-15 dB — 문자 그대로 **항등식**).

**구는 즉시 갈라놓습니다**: 위상계수 2k→k 는 +6 dB, obliquity 제거는 −30 dB.

### ⚠️ 그런데 구도 못 잡는 버그가 하나 있습니다
**위상 부호 반전**은 구에서도 통과합니다. 부호를 뒤집으면 E → E\* 이고, **σ ∝ |E|² 는 켤레에 불변**이라 *어떤 형상의 모노스태틱 RCS 시험으로도* 검출할 수 없습니다.
그래서 **위상 기울기 dφ/dR** 시험을 추가했습니다: 정상 +1.00 · 2k→k +0.50 · **부호 반전 −1.00**.

> 🔴 위상 부호는 **마이크로도플러의 도플러 부호**를 결정합니다(report3). RCS 숫자는 전부 멀쩡한 채로 도플러만 뒤집힐 수 있었습니다. **σ 검증만으로는 부족합니다.**

### (b) 구 ↔ 해석적 PO: 커널은 맞다 · 점간격 λ/7 은 계통적으로 낮다
![convergence](outputs/figures/report6_po_convergence.png)

구의 **해석적 PO** 적분과 대조하면 이산 PO 는 모든 점간격에서 |Δ| ≤ 0.04 dB 로 재현합니다(λ/4 에서도). **커널은 맞습니다.** 다만 드론에서는 λ/7 이 λ/30 대비 **단조롭게, 항상 낮은 쪽**입니다(−0.02 ~ −0.30 dB) — **알려진 편향**입니다.

> 📌 이 두 그림은 **일부러 PO 를 시험하는 그림**입니다. SBR 로 바꾸면 실험 자체가 사라지므로 **그대로 둡니다.**

## 9. 널(null) — 위치는 믿고, 깊이는 인용하지 마라 (기존, 유지)

![nulls](outputs/figures/report6_nulls.png)

1. **널의 위치는 이산화에 안정적**입니다 (λ/7 → λ/20 에서 어떤 널도 방위 격자 한 칸 이상 안 움직임).
2. **그러나 깊이는 신뢰할 수 없습니다** — 같은 널의 깊이가 이산화에 따라 수 dB ~ 10 dB 요동하고 수렴하지 않습니다. 널 깊이는 **거의 상쇄의 잔차**이고, 잔차는 표적이 아니라 **이산화 오차**가 정합니다.
3. **실제 레이더는 그 바늘을 못 봅니다** — 100 MHz 대역평균 + 3° 각도창을 넣으면 최저 널이 20 dB 이상 올라옵니다. 반면 **로브 피크와 방위평균은 불변** — 앵커링 수치는 안전합니다.

> → report2 의 방위 패턴은 **"레이더가 실제로 보는 값"**(대역평균 + 각도창)으로 그리고, **널 깊이 숫자는 어디에도 인용하지 않습니다.**

In [ ]:
# (선택 실행) 핵심 실험 재현 — GPU 필요
#   !python benchmark/verify_rt_rays.py      # §2 — 4억 발
#   !python benchmark/verify_rt_no_rcs.py    # §3~§5
import json
R = json.load(open('outputs/rt_ray_budget.json'))
S = json.load(open('outputs/report6_sbr.json'))

print('§2  광선을 늘리면? — 정답선 %.1f dB (SBR sigma %.1f dBsm)'
      % (R['truth']['ratio_db_truth'], R['truth']['full_dbsm']))
for r in R['A_ray_sweep']:
    if r['n_paths']:
        print(f"    {r['spp']/1e6:5.0f}M rays -> {r['n_mean']:5.1f} paths | "
              f"incoherent {r['incoh_db']:+7.2f} +/- {r['incoh_sd']:.2f} dB | "
              f"coherent {r['coh_db']:+7.2f} dB")
print('    => 수렴하지 않는다. sigma 는 적분에서 나온다.')

print('\n§6  가림(occlusion)의 대가 — PO -> SBR, 방위평균 dBsm @ el=15')
for k, v in S['compare'].items():
    print(f"    {k:10s} PO {v['po_el15']:+7.2f} -> SBR {v['sbr_el15']:+7.2f} "
          f"({v['occl_el15']:+.2f} dB) | 다중반사 {v['multibounce_db']:+.2f} dB")

print('\n§7  메쉬 외형 버그 — 수정 전 -> 공식')
for k, v in S['envelope'].items():
    print(f"    {k:10s} 높이 {v['h_old_mm']:6.1f} -> {v['h_new_mm']:6.1f} mm "
          f"({v['h_err_pct']:+5.1f}%) | sigma(el15) {v['d_el15']:+.2f} dB")

## 10. 챔버 바닥 — 클러터는 **죽은 파라미터**였고, 진짜 위협은 **유령**이었다

> 발표자료 리뷰에서 나온 건 **용어 지적 하나**였습니다: *"챔버를 anechoic 이라 부르는데 바닥이 반사면이면 semi-anechoic 이 맞다."* 오타 수정처럼 보였습니다. **아니었습니다.**

### (a) 바닥은 실재하고, **Sionna RT 는 처음부터 보고 있었다**

![floor](outputs/figures/report6_floor_geom.png)

흡수체는 **벽 4면 + 천장**에만 있습니다. **바닥은 반사성 콘크리트**입니다(`materials.py` → ITU `concrete`). 그러면 이 방은 정의상 anechoic 이 아니라 **semi-anechoic** 입니다.

거울상으로 펴서 닫힌형으로 계산한 바닥 반사(**+19.3 ns / −14.7 dB**)를 **Sionna RT 가 실측한 경로 목록에서 그대로 찾았습니다** — 지연 0.0 ns, 진폭 **−0.02 dB** 일치. (report1 이 독립적으로 같은 값을 재확인했습니다.)

### (b) 그런데 왜 아무 수치도 안 틀렸나 — 클러터가 **아무것도 결정하지 못하는 자리**에 있었다

![dead](outputs/figures/report6_clutter_dead.png)

우리 가정의 최강 탭은 **−26.0 dB**, RT 실측 최강은 **−9.8 dB** — **16 dB 과소가정**입니다. 그런데 클러터 진폭을 **직접파보다 14 dB 세게** 넣어도 SCR 이 **1e-9 dB** 움직입니다. 물리가 아니라 **구조**입니다 — 정적 클러터는 **삼중으로 차단**돼 있었습니다:

1. **ECA 사영** — 정적 클러터는 도플러 항이 없는 지연된 기준신호의 선형결합입니다. ECA 의 기저가 정확히 그 부분공간이라 **진폭과 무관하게 정확히 0** 으로 지웁니다. **사영은 진폭을 보지 않습니다.**
2. 잔류는 **0-도플러 행**에 앉습니다.
3. **그 행이 삭제됩니다** (`det[zd, :] = False`).

> **이것이 무너뜨리는 것**: report5 §D 의 *"RT 와 Analytic 이 일치하므로 클러터 모델이 검증됐다"* 는 **검증이 아니라 항등식**이었습니다 — 어떤 클러터 모델을 넣어도 일치할 수밖에 없었습니다. (report5 가 이 사실을 **잔향 ×0 / ×1 / ×10 → SCR 43.140551 dB 소수점 6자리까지 동일**로 재확인했습니다.)

> **오해 금지**: "정적 클러터는 무해하다"가 아닙니다. **이 모델이 그걸 검증할 능력이 없다**는 뜻입니다. 실제 ECA 는 유한 동적범위·클러터 도플러퍼짐 때문에 이렇게 완벽하지 않고, 그 한계는 **아직 모델에 없습니다.**

### (c) 진짜 위협 — **표적 경유 바닥 유령**

![ghost](outputs/figures/report6_ghost_cfar.png)

**TX → 표적 → 바닥 → RX.** 이 경로는 표적을 거치므로 **표적과 함께 도플러가 실립니다.** 정적 클러터와 달리 **ECA 의 영공간 밖**이고 0-도플러 행도 아닙니다 → **안 지워지고 CFAR 에 그대로 노출됩니다.**
유령의 위치(표적 뒤 **+3.5 m**)와 세기(**−18 dB**)는 파형과 거의 무관합니다 — 순전히 기하이고, 갈리는 건 **거리분해능**뿐입니다(5G 의 ΔRb ≈ 3.1 m 는 그걸 분해하고, LTE 의 16.7 m 는 못 합니다).

### ⚠️ (d) **미해결 — 유령이 '가짜 표적'이 되는가에 대해 두 리포트가 갈립니다**

| | 결론 | 근거 |
|---|---|---|
| **report5 §E** (유령 매트릭스) | 5G 에서 **P(가짜표적) = 100%** | 유령 셀이 CFAR 임계보다 **+15.9 dB** 위, 매 시행 검출 |
| **report4** (대조군 추가) | **P(가짜표적) = 0%** | **바닥을 꺼도** 같은 셀의 히트율이 **똑같이 100%** → 그 히트는 유령이 아니라 **표적 자신의 거리부엽** |

report4 는 유령이 표적에서 **1.4 거리샘플**밖에 안 떨어져 있어 표적 자신의 응답에 묻히고, 바닥을 켜도 유령 셀이 **+1.5 dB** 밖에 안 오른다고 측정했습니다. report5 는 대조군 없이 유령 셀의 검출률만 쟀습니다.

> **이 노트북은 어느 쪽 편도 들지 않습니다.** 두 실험은 **다른 것을 재고 있습니다** — report5 는 *"그 셀에 검출이 뜨는가"*, report4 는 *"그 검출이 유령 때문인가"*. **대조군이 있는 쪽(report4)이 방법론적으로 더 강하지만**, 두 실험의 CFAR 창·셀 정의가 같은지 아직 대조하지 않았습니다. → §11 리스크 표에 **열린 항목**으로 올립니다.

> **어느 쪽이든 살아남는 사실**: 유령은 **실재하고, ECA 를 통과합니다**(도플러가 실려 있으니까). 그리고 그건 **정적 클러터가 못 하는 일**입니다.

## 11. 판정 & 남은 리스크

![hybrid](outputs/figures/report6_hybrid.png)

### 하이브리드(표적 = SBR/PO, 환경 = RT)는 정당한가 — **그렇다. 단, 이유를 정확히 말해야 한다.**

정당한 이유는 **"Sionna RT 의 기본 path solver 에 산란적분 단계가 없어 표적 σ 가 창발하지 않기 때문"** 입니다. 우리 설정 실수나 표본 부족이 아닙니다 — §2 가 광선 16배로 그걸 **측정으로** 못박았습니다.

**정당화에 쓰면 안 되는 근거** (적대적 검증에서 깨진 것들):

- ❌ *"RCS 는 레이트레이싱에서 창발하지 않는다"* — **거짓**입니다. **SBR 은 레이트레이싱이고, σ 를 계산합니다**(§6). 참인 명제는 좁습니다: **"산란적분 단계가 없는 전파용 path solver 에서는 창발하지 않는다."**
- ❌ *"광선이 부족해서다"* — **거짓**입니다(§2: 4억 발에도 수렴 안 함).
- ❌ *"3GPP 가 지지하는 표준 관행"* — 범주오류입니다(3GPP ISAC 의 RCS 파라미터화는 **통계적 채널모델 안**의 이야기).
- ❌ *"정공법이 아예 없다"* — **custom PathSolver** 로 in-Sionna 산란적분을 넣는 길은 존재합니다. 우리가 안 하는 이유는 **비용**이지 부재가 아닙니다.

### 리스크 표 — **해결된 것 / 정량화된 것 / 아직 모르는 것**

| 리스크 | 상태 |
|---|---|
| ~~PO 에 가림(self-shadowing)이 없다~~ | ✅ **해결** — SBR(§6). 광선이 조명면을 정한다. σ 가 1.5~7.1 dB 내려갔다 |
| ~~PO 는 오목부 다중반사를 못 한다~~ | ✅ **정량화됨** — SBR 3-bounce: **-0.33 ~ +0.35 dB**. 무시할 만하다 |
| ~~메쉬 외형이 제원과 어긋난다~~ | ✅ **해결** — 공식 L×W×H 맞춤 + **trimesh 게이트**(§7) |
| ~~두 엔진이 재질을 다르게 본다~~ | ✅ **해결** — `materials.py` 단일 진리원(§7) |
| **PO/SBR 의 절대 RCS 불확실도** | 🔴 **여전히 모릅니다.** SBR 은 셸 안(배터리·PCB)에 광선이 못 닿아 **하한**이고, PO 는 가림이 없어 **상한**입니다. **참값은 그 사이** — 실측 앵커링 전에는 미확정 |
| SBR 광선격자의 잡음 | ⚠️ 작업 격자(λ/12)에서 구 오차 **+1.45 dB** — 계통 편향이 아니라 이산화 잡음. λ/16+ 에서 **≤ 0.58 dB** |
| PO 점간격 λ/7 의 계통 편향 | ⚠️ −0.02 ~ −0.30 dB (λ/30 대비). 소폭이지만 알려진 편향 |
| 널 깊이 | 🔴 **인용 불가** (이산화 의존) |
| **ECA 의 이상화** (§10) | 🔴 유한 동적범위·클러터 도플러퍼짐이 **모델에 없습니다.** 정적 클러터가 실제로 무해한지는 **모릅니다** |
| **바닥 유령의 오검출률** (§10d) | 🔴 **report4(0%) 와 report5(100%) 가 갈립니다.** 대조군 유무의 차이 — 동일 CFAR 규약으로 재대조 필요 |
| 흡수체 −25 dB | ⚠️ **실측 아님.** 평판 단일면은 ≈−5 dB 이고 −25 dB 는 피라미드 다중반사로 달성되는 **설계 목표** |
| 챔버 RT 경로예산 절단 | ⚠️ `max_num_paths_per_src` 에 챔버 총경로수가 점근 중 — 확인 필요 |

### 다음 단계
1. **PO/SBR 절대 불확실도** — 이게 남은 가장 큰 구멍입니다. 셸 투과(반투명 셸)를 SBR 에 넣거나, 실측 RCS 로 앵커링해야 SBR(하한)과 PO(상한) 사이가 좁혀집니다.
2. **유령 오검출 재대조** — report4 의 대조군 규약을 report5 §E 매트릭스에 그대로 적용해 두 결론을 합칩니다.
3. **ECA 현실화** — 유한 동적범위(ADC 비트)·클러터 도플러퍼짐을 넣어야 정적 클러터가 **비로소 의미를 갖습니다.**
4. (선택) **in-Sionna RCS** — custom PathSolver 에 산란적분을 넣는 정공법. 하이브리드가 이미 정당하므로 급하지 않습니다.

---

> **정리 — 이 리포트가 알아낸 것**
> - 챔버의 −5 dB '일치' 는 **오분류**였고,
> - 평판 검증은 **항등식**이었으며,
> - 널 깊이는 **인용 불가**였고,
> - 클러터 모델은 **아무것도 결정하지 못하는 자리**에 있어서 16 dB 를 틀리고도 아무도 못 알아챘고,
> - 정말로 결정하는 것(**바닥 유령**)은 모델에 아예 없었으며,
> - "광선을 더 쏘면 된다" 는 **4억 발로 반증**됐고,
> - 기존 PO 는 σ 를 **1.5~7.1 dB**, 마이크로도플러를 **17.4 dB** 과대평가하고 있었으며,
> - 드론은 **47% 까지 납작했습니다.**
>
> **그리고 이 대부분은 — 발표자료의 용어 오타 하나와 "광선을 더 쏘면 되지 않나" 라는 반문을 끝까지 따라간 결과** 나왔습니다.